# 🎯 Centralized Deep Learning Training Pipeline (Scenario E1)
## Intrusion Detection on CICIoT2023 with TabularIoTMLP & Multi-Class Focal Loss

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/centralized_training_pipeline.ipynb)

This notebook provides the complete, publication-grade **Centralized Training Pipeline (Scenario E1)** for the **FL-IoT-IDS** project. It trains the lightweight `TabularIoTMLP` neural network on centralized network flow data to establish the **empirical performance upper bound ceiling** against which all distributed Federated Learning simulations are benchmarked.

### Pipeline Architecture:
1. **Environment Setup & Seed:** Git pull repository, install exact dependencies, and set global deterministic seed (`42`).
2. **Persistent Dataset Mounting:** Load the real 991 MB `merged_CICIOT2023_data.csv` directly from Google Drive.
3. **Leak-Free Tabular Preprocessing:** Stratified train/val/test split (70/10/20), $\pm\infty$ cleaning, training median imputation, non-negative physical clipping, $\log(1+x)$ heavy-tail compression, and `RobustScaler` IQR normalization.
4. **Class Imbalance Strategy:** Inverse-frequency class weights ($w_c$) paired with mathematically exact `MultiClassFocalLoss` (focusing parameter $\gamma=2.0$) to prevent dominant attack swamping.
5. **Training Execution:** `CentralizedTrainer` with AdamW optimizer, gradient clipping ($5.0$), live batch progress bar (`tqdm`), validation tracking, and patience-based early stopping.
6. **Rigorous Evaluation:** Comprehensive holdout evaluation reporting Macro-F1, isolated Minority Recall (`Web-based`, `Brute-force`), and row-normalized Confusion Matrix.
7. **Artifact Archival:** Automatically saves model checkpoints and metric reports to Google Drive.

---
### 🛠️ Step 1: Environment Setup, Synchronization & Deterministic Seed
We verify GPU acceleration, pull the latest code from `https://github.com/NVKQ2022/FedLearning.git`, install dependencies, and enforce a **bit-level deterministic seed** across Python, NumPy, and PyTorch (CPU & CUDA) for 100% scientific reproducibility.

In [ ]:
# 1. Hardware acceleration verification
import torch
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


In [ ]:
# 2. Clone or pull the latest repository codebase
import os, sys, subprocess

repo_url = "https://github.com/NVKQ2022/FedLearning.git"
repo_dir = "/content/FedLearning"

if not os.path.exists(repo_dir):
    print(f"Cloning repository from {repo_url}...")
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
else:
    print("Repository already present. Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", repo_dir, "reset", "--hard", "origin/main"], check=True)

os.chdir(repo_dir)
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

# Purge any cached in-memory src modules so fresh code is loaded
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

print(f"✅ Working directory set to: {os.getcwd()}")


In [ ]:
# 3. Install dependencies from requirements.txt
!pip install -q -r requirements.txt
print("✅ Dependencies verified and installed.")


In [ ]:
# 4. Enforce bit-level deterministic seed for scientific reproducibility
from src.utils.seed import set_seed

# Change this integer seed (e.g. 42, 123, 999) to run exact ablation replications
EXPERIMENT_SEED = 42
set_seed(EXPERIMENT_SEED)
print(f"✅ Deterministic random seed successfully set to: {EXPERIMENT_SEED}")


---
### 💾 Step 2: Google Drive Mounting & Dataset Acquisition
We mount Google Drive to load the real **991.00 MB** dataset (`merged_CICIOT2023_data.csv`).

* **Target Path in Google Drive:** `/content/drive/MyDrive/NguyenVietKyQuanKLTN/datasets/merged_CICIOT2023_data.csv`
* **Local SSD Fast Cache:** `/content/FedLearning/datasets/CICIOT2023/merged_CICIOT2023_data.csv`

In [ ]:
# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Drive mount skipped or not running on Colab:', e)

import os, shutil

target_csv_path = "/content/FedLearning/datasets/CICIOT2023/merged_CICIOT2023_data.csv"
os.makedirs(os.path.dirname(target_csv_path), exist_ok=True)

# Primary Google Drive path configured for thesis project
gdrive_csv = "/content/drive/MyDrive/NguyenVietKyQuanKLTN/datasets/merged_CICIOT2023_data.csv"

# Fallback check if stored under alternative Drive folder
if not os.path.exists(gdrive_csv):
    alt_gdrive_csv = "/content/drive/MyDrive/FedLearning/merged_CICIOT2023_data.csv"
    if os.path.exists(alt_gdrive_csv):
        gdrive_csv = alt_gdrive_csv

if os.path.exists(gdrive_csv):
    gdrive_size = os.path.getsize(gdrive_csv) / (1024**2)
    # Overwrite if target is missing or if it's the small 21 MB synthetic file
    if not os.path.exists(target_csv_path) or os.path.getsize(target_csv_path) < 100 * 1024 * 1024:
        print(f"Copying real 991 MB dataset from Google Drive to local SSD ({gdrive_size:.2f} MB)...")
        shutil.copy(gdrive_csv, target_csv_path)
    print(f"✅ REAL CICIoT2023 dataset ready on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
elif os.path.exists(target_csv_path) and os.path.getsize(target_csv_path) >= 100 * 1024 * 1024:
    print(f"✅ REAL CICIoT2023 dataset already present on local SSD: {os.path.getsize(target_csv_path) / (1024**2):.2f} MB")
else:
    print(f"⚠️ Real dataset NOT found at '{gdrive_csv}'.")
    print("Please verify that merged_CICIOT2023_data.csv exists in Google Drive under NguyenVietKyQuanKLTN/datasets/ !")


---
### 🔄 Step 3: Leak-Free Preprocessing & Balanced Class Weights
We use `load_and_preprocess_ciciot2023` to perform:
* **Stratified train/val/test split (70 / 10 / 20)** strictly before computing statistics.
* **Outlier & skew mitigation:** `RobustScaler` (median & IQR) and $\log(1+x)$ compression.
* **Class weight calculation:** Inverse-frequency class weights ($w_c = \frac{N}{C \times N_c}$) to penalize errors on rare attack vectors.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
import numpy as np

# Set sample_size=100000 for rapid training (or sample_size=None to train on the full 5.11M rows)
SAMPLE_SIZE = 100000
BATCH_SIZE = 128

print(f"Loading and preprocessing CICIoT2023 (sample_size={SAMPLE_SIZE:,})...")
data = load_and_preprocess_ciciot2023(
    csv_path=target_csv_path,
    test_size=0.2,       # 20% holdout test set
    val_size=0.1,        # 10% validation set
    sample_size=SAMPLE_SIZE,
    scaler_type="robust",
    batch_size=BATCH_SIZE,
    random_state=EXPERIMENT_SEED
)

class_weights = compute_balanced_class_weights(data["y_train"], num_classes=8)
class_names = data["class_names"]
minority_classes = ["Web-based", "Brute-force"]

print("\n--- Dataset Partition Summary ---")
print(f"Training set:   {len(data['X_train']):,} samples ({len(data['train_loader']):,} batches)")
print(f"Validation set: {len(data['X_val']):,} samples ({len(data['val_loader']):,} batches)")
print(f"Holdout Test:   {len(data['X_test']):,} samples ({len(data['test_loader']):,} batches)")
print("Detected Classes:", class_names)
print("Inverse Class Weights:", np.round(class_weights, 3))


---
### 🧠 Step 4: Model Architecture & Optimization Suite
We instantiate:
* **`TabularIoTMLP`:** 39 input features $\to 128 \to 64 \to 8$ classes with Dropout ($0.2$) and Kaiming normal initialization (13,896 parameters, ~54.28 KB).
* **`MultiClassFocalLoss`:** Decoupled probability calculation with focusing parameter $\gamma=2.0$ and class weights tensor $\alpha_t$.
* **`AdamW`:** Decoupled weight decay ($10^{-4}$) preventing weight explosion during extended training.

In [ ]:
from src.models.mlp import TabularIoTMLP
from src.losses.focal_loss import build_loss_function
from src.optimizers.optimizer import build_optimizer

# 1. Initialize Neural Network Backbone
model = TabularIoTMLP(
    input_dim=data["input_dim"],  # 39
    hidden_dims=(128, 64),
    num_classes=data["num_classes"],  # 8
    dropout_rate=0.2
).to(device)

print(f"Model Trainable Parameters: {model.get_num_parameters():,}")
print(f"Model Weight Payload Size:  {model.get_model_size_kb():.2f} KB")

# 2. Build Focal Loss with inverse-frequency class weights
criterion = build_loss_function(
    loss_type="focal_loss",
    class_weights=class_weights,
    gamma=2.0,
    device=device
)

# 3. Build AdamW Optimizer
optimizer = build_optimizer(
    model=model,
    optimizer_type="adamw",
    lr=1e-3,
    weight_decay=1e-4
)


---
### 🏋️ Step 5: Centralized Model Training with Live Progress & Early Stopping
We use `CentralizedTrainer` with gradient clipping ($5.0$), live mini-batch progress bars (`tqdm`), and patience-based validation checkpointing (`patience=4`), automatically restoring the best model weights upon completion.

In [ ]:
from src.training.trainer import CentralizedTrainer

EPOCHS = 15
PATIENCE = 4

trainer = CentralizedTrainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_grad_norm=5.0
)

print(f"Starting Centralized Training (Epochs={EPOCHS}, Patience={PATIENCE})...")
history = trainer.fit(
    train_loader=data["train_loader"],
    val_loader=data["val_loader"],
    epochs=EPOCHS,
    patience=PATIENCE,
    verbose=True
)

print(f"\n✅ Training complete! Best validation loss achieved: {trainer.best_val_loss:.4f}")


---
### 📈 Step 6: Training Learning Curves
We visualize loss convergence and accuracy progression across training epochs.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

# 1. Loss Curve
axes[0].plot(epochs_range, history["train_loss"], 'o-', label="Train Loss", color="#e74c3c", linewidth=2)
axes[0].plot(epochs_range, history["val_loss"], 's--', label="Val Loss", color="#2980b9", linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Focal Loss", fontsize=12)
axes[0].set_title("Centralized Training & Validation Loss", fontsize=13, fontweight="bold")
axes[0].legend(loc="upper right")

# 2. Accuracy Curve
axes[1].plot(epochs_range, [acc * 100 for acc in history["train_acc"]], 'o-', label="Train Acc", color="#27ae60", linewidth=2)
axes[1].plot(epochs_range, [acc * 100 for acc in history["val_acc"]], 's--', label="Val Acc", color="#8e44ad", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy (%)", fontsize=12)
axes[1].set_title("Centralized Accuracy Trajectory", fontsize=13, fontweight="bold")
axes[1].legend(loc="lower right")

plt.tight_layout()
os.makedirs("reports/figures", exist_ok=True)
plt.savefig("reports/figures/centralized_learning_curves.png", dpi=300, bbox_inches="tight")
plt.show()


---
### 🏆 Step 7: Comprehensive Holdout Test Evaluation
We evaluate the restored best model on the completely unseen holdout test set (20% of data, zero leakage).
We inspect:
* **Overall Accuracy** & **Macro-F1 Score**
* **Per-Class Precision, Recall, and F1**
* **Minority Class Recall Isolation** (`Web-based`, `Brute-force`)
* **Row-Normalized Confusion Matrix Heatmap**

In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive

test_results = evaluate_comprehensive(
    model=model,
    dataloader=data["test_loader"],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print("=" * 60)
print("🏆 SCENARIO E1 (CENTRALIZED UPPER BOUND) FINAL RESULTS")
print("=" * 60)
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print("-" * 60)
print("🎯 Minority Class Recall (Stealth Penetration Attacks):")
for cls, rec in test_results["minority_recall"].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print("-" * 60)
print("📊 Per-Class Breakdown:")
for cname in class_names:
    f1 = test_results["per_class_f1"][cname] * 100
    rec = test_results["per_class_recall"][cname] * 100
    prec = test_results["per_class_precision"][cname] * 100
    supp = test_results["support"][cname]
    print(f"   {cname:<12} | F1: {f1:5.1f}% | Recall: {rec:5.1f}% | Prec: {prec:5.1f}% | Support: {supp:,}")
print("=" * 60)


In [ ]:
# Plot Row-Normalized Confusion Matrix Heatmap
plt.figure(figsize=(10, 8))
cm = test_results["confusion_matrix"]
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True
)
plt.title("Scenario E1: Normalized Confusion Matrix (Centralized Baseline)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Class", fontsize=12)
plt.ylabel("True Class", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
cm_path = "reports/figures/centralized_confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()


---
### 💾 Step 8: Export Model Checkpoints & Reports to Google Drive
We serialize the best model weights (`.pth`), preprocessor state (`.pkl`), and evaluation JSON report directly to `/content/drive/MyDrive/NguyenVietKyQuanKLTN/` so your findings are permanently archived.

In [ ]:
import json

# 1. Save locally
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)

model_save_path = "checkpoints/centralized_e1_model.pth"
torch.save(model.state_dict(), model_save_path)

preprocessor_save_path = "checkpoints/preprocessor.pkl"
data["preprocessor"].save(preprocessor_save_path)

metrics_export = {k: v for k, v in test_results.items() if k != "confusion_matrix"}
metrics_file = "reports/centralized_e1_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(metrics_export, f, indent=4)

print(f"✅ Model weights saved to: {model_save_path}")
print(f"✅ Preprocessor saved to:   {preprocessor_save_path}")
print(f"✅ Metrics report saved:   {metrics_file}")

# 2. Archive to Google Drive folder for thesis
gdrive_dest = "/content/drive/MyDrive/NguyenVietKyQuanKLTN/checkpoints"
gdrive_reports = "/content/drive/MyDrive/NguyenVietKyQuanKLTN/reports"

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(gdrive_dest, exist_ok=True)
    os.makedirs(gdrive_reports, exist_ok=True)
    shutil.copy(model_save_path, os.path.join(gdrive_dest, "centralized_e1_model.pth"))
    shutil.copy(preprocessor_save_path, os.path.join(gdrive_dest, "preprocessor.pkl"))
    shutil.copy(metrics_file, os.path.join(gdrive_reports, "centralized_e1_metrics.json"))
    shutil.copy("reports/figures/centralized_learning_curves.png", os.path.join(gdrive_reports, "centralized_learning_curves.png"))
    shutil.copy("reports/figures/centralized_confusion_matrix.png", os.path.join(gdrive_reports, "centralized_confusion_matrix.png"))
    print(f"\n🎉 All checkpoints, plots & reports permanently archived to Google Drive: {gdrive_dest}")
else:
    print("Google Drive not mounted. Results saved in local Colab session.")
